# Lab 4 — FastAPI Scoring API

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Goals

1. Wrap a trained KNN pipeline in a **FastAPI** application.
2. Define **Pydantic** request/response models for typed JSON I/O.
3. Expose `GET /health` and `POST /predict` endpoints.
4. Verify the API with **`TestClient`** (no live server required).

> **Quick check:** `GET /health` → **200** · `POST /predict` → **200** · `default_probability` ≈ **0.2**, `default_label` = **0**




**Day 4 flow:** Labs 1–3 KNN → **Lab 4 (you are here)** FastAPI → Lab 5 FeatureTools → Lab 6 MLflow + DVC.

## From notebook to production API

| Layer | Role |
|-------|------|
| **Trained model** | `Pipeline` with scaler + KNN — same artifact you'd pickle or log to MLflow |
| **Pydantic models** | Validate incoming JSON (types, ranges) before scoring |
| **FastAPI routes** | HTTP interface: health check + predict |
| **TestClient** | In-process HTTP tests — no `uvicorn` needed for the lab |

```text
Client JSON  →  LoanRequest  →  DataFrame  →  model.predict_proba  →  LoanPrediction JSON
```

## REST API cheat sheet

| Method | Path | Purpose |
|--------|------|--------|
| GET | `/health` | Liveness probe |
| POST | `/predict` | Score one loan |
| GET | `/docs` | Swagger UI (live server) |

---

## 1. Train the scoring model

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]
X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)
model.fit(X_train, y_train)
print(f"model trained on {len(X_train)} loans (k=5)")


### 1b. Verify model predicts on a training row

In [ ]:
train_sample = X_train.iloc[[0]]
pred = model.predict(train_sample)[0]
proba = model.predict_proba(train_sample)[0]
print(f"sample train prediction: {pred}, P(default)={proba[1]:.3f}")


---

## 2. Pydantic request and response models

In [ ]:
from pydantic import BaseModel, Field

class LoanRequest(BaseModel):
    loan_amnt: float = Field(..., gt=0)
    int_rate: float = Field(..., ge=0)
    annual_inc: float = Field(..., gt=0)
    dti: float = Field(..., ge=0)
    installment: float = Field(..., gt=0)

class LoanPrediction(BaseModel):
    default_probability: float
    default_label: int

sample_request = LoanRequest(
    loan_amnt=15000,
    int_rate=12.5,
    annual_inc=65000,
    dti=18.0,
    installment=450.0,
)
display(sample_request.model_dump())



`Field(..., gt=0)` rejects invalid inputs (e.g. negative loan amount) **before** the model runs — a key API safety layer.

### 2b. JSON schema preview

In [ ]:
import json
print(json.dumps(LoanRequest.model_json_schema(), indent=2)[:500], "...")


---

## 3. Define the FastAPI app

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Lending Club Scoring API")

@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}

@app.post("/predict", response_model=LoanPrediction)
def predict(loan: LoanRequest) -> LoanPrediction:
    features = pd.DataFrame(
        [[loan.loan_amnt, loan.int_rate, loan.annual_inc, loan.dti, loan.installment]],
        columns=NUMERIC_FEATURES,
    )
    proba = float(model.predict_proba(features)[0][1])
    label = int(proba >= 0.5)
    return LoanPrediction(default_probability=round(proba, 4), default_label=label)

print("routes:", [route.path for route in app.routes])



### 3b. Route summary

In [ ]:
for route in app.routes:
    methods = getattr(route, "methods", set())
    print(f"{methods} {route.path}")


---

## 4. Test with `TestClient` (no live server)

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

health = client.get("/health")
sample = {
    "loan_amnt": 15000,
    "int_rate": 12.5,
    "annual_inc": 65000,
    "dti": 18.0,
    "installment": 450.0,
}
response = client.post("/predict", json=sample)

print("Lab 4 — FastAPI scoring API")
print(f"GET /health -> {health.status_code} {health.json()}")
print(f"POST /predict -> {response.status_code}")
print(f"response body: {response.json()}")


### 4b. Response model validation

In [ ]:
parsed = LoanPrediction.model_validate(response.json())
print("Parsed response:", parsed)


---

## 5. Change the request — observe probability shift

In [ ]:
riskier = {
    "loan_amnt": 25000,
    "int_rate": 22.0,
    "annual_inc": 40000,
    "dti": 28.0,
    "installment": 850.0,
}
safer = sample.copy()

rows = []
for label, body in [("baseline", safer), ("riskier", riskier)]:
    r = client.post("/predict", json=body)
    rows.append({"profile": label, **r.json()})

display(pd.DataFrame(rows))


Higher `int_rate` and `dti` with lower income typically push `default_probability` up — the API returns a different score without retraining.

### 5b. Probability threshold

In [ ]:
for row in rows:
    p = row["default_probability"]
    label = 1 if p >= 0.5 else 0
    print(f"{row['profile']}: P(default)={p:.4f} → label {label}")


---

## 6. Validation error example

In [ ]:
bad = sample.copy()
bad["loan_amnt"] = -1000
bad_response = client.post("/predict", json=bad)
print(f"invalid loan_amnt -> HTTP {bad_response.status_code}")
print(bad_response.json())


Pydantic returns **422 Unprocessable Entity** for invalid input — the model never runs on bad data.

### 6b. Missing field

In [ ]:
incomplete = {"loan_amnt": 15000, "int_rate": 12.5}
missing_resp = client.post("/predict", json=incomplete)
print(f"missing fields -> HTTP {missing_resp.status_code}")


---

## 7. OpenAPI / Swagger (live server)

When running with `uvicorn`, open `http://127.0.0.1:8000/docs` for interactive API documentation. FastAPI auto-generates the schema from Pydantic models.

---

## 8. Live server (instructor demo)

To run outside the notebook:

```bash
cd hands-on/04-distance-mlops/output
uvicorn lab04_fastapi_scoring_api:app --reload
```

Then POST to `http://127.0.0.1:8000/predict` with the same JSON body.

---

## 9. MLOps connection

Lab 6 logs this same pipeline to **MLflow**. In production you would:

1. Load the model artifact from MLflow Model Registry.
2. Mount it in the FastAPI `predict` handler.
3. Use `/health` for Kubernetes liveness probes.

---

## 10. Try it yourself

POST a loan with `int_rate=28` and compare `default_probability` to the baseline sample.

In [ ]:
# Your code here (optional)


In [ ]:
high_rate = sample.copy()
high_rate["int_rate"] = 28.0
r_high = client.post("/predict", json=high_rate)
print("baseline:", response.json())
print("high int_rate:", r_high.json())


### 10b. Batch scoring pattern (loop)

In [ ]:
profiles = [safer, riskier]
batch = []
for name, body in [("baseline", safer), ("riskier", riskier)]:
    r = client.post("/predict", json=body)
    batch.append({"profile": name, **r.json()})
display(pd.DataFrame(batch))


### 10c. Response headers

In [ ]:
print("Content-Type:", response.headers.get("content-type"))
print("Status:", response.status_code)


### 10d. Model feature order contract

In [ ]:
print("API expects features in order:", NUMERIC_FEATURES)
print("Mismatch in column order would silently score wrong inputs — document the contract.")


### 10e. Curl example (live server)

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"loan_amnt":15000,"int_rate":12.5,"annual_inc":65000,"dti":18,"installment":450}'
```

### 10f. API versioning (concept)

Production APIs version routes (`/v1/predict`) so you can deploy a new model without breaking existing clients.

### 10g. Logging requests (concept)

In [ ]:
# In production: log request_id, latency, and score — not raw PII without policy
print("Sample request logged:", sample)


### 10h. Async vs sync (concept)

FastAPI supports `async def` routes for I/O-bound work. sklearn `predict` is CPU-bound and runs synchronously here — for high throughput, use a worker queue or model server.

### 10i. OpenAPI title and version

In [ ]:
print("App title:", app.title)
print("OpenAPI schema keys:", list(app.openapi().keys())[:4])


### 10j. Rate limiting (concept)

Public scoring APIs need **rate limits** and **authentication** (API keys, OAuth) before production traffic.

---

## 11. Checkpoint summary

In [ ]:
body = response.json()
assert health.status_code == 200
assert health.json() == {"status": "ok"}
assert response.status_code == 200
assert body["default_label"] == 0
assert abs(body["default_probability"] - 0.2) < 0.05
print("Numbers match — you're good.")



---

## Reflection

1. Why separate `/health` from `/predict` in production deployments?
2. What would you add before exposing this API on the public internet?
3. How does Lab 6 (MLflow) relate to versioning the model behind this API?

**Previous:** [Lab 3 — Choose K](lab03_choose_k.ipynb)  
**Next:** [Lab 5 — FeatureTools auto FE](lab05_featuretools_auto_fe.ipynb)